# 08 · Removing the query-string shortcut, on a frozen benchmark

Notebook 07's CNN scored 0.975 on its own test set but flagged so much of my real browsing that it was useless. The likely cause: 87% of the legit training URLs come from sitemaps, which almost never contain `?`, so the model learned "query string = phishing". A probe with `index.html?id=...` scored 0.892.

Two changes here:

1. **Freeze the benchmark.** Until now it was rebuilt every time, and my growing Chrome history and fresh OpenPhish downloads moved v1's scores (1.2% → 5.8% history false alarms) without the model changing. From now on every notebook scores the same saved URLs.
2. **Strip query strings** (everything after `?`) before training and scoring, so the shortcut doesn't exist. Both full-URL models (TF-IDF and char-CNN) are trained with and without queries, so the effect is measured directly.

**The prediction to test:** if the query shortcut was the problem, the no-query models should raise far fewer false alarms on my history, especially on history URLs that contain `?`.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import torch
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

from phishdetect.data import PROJECT_ROOT
from phishdetect.fullurl import normalize_full_url, shallow_stats, FullUrlTfidf
from phishdetect.charcnn import CharCNNClassifier, best_device
from phishdetect.detector import PhishDetector
from phishdetect.frozen import FROZEN_PATH, freeze_benchmark, load_benchmark
from phishdetect.benchmark import defang

pd.set_option("display.max_colwidth", 110)
RANDOM_STATE = 42
TARGET_FA = 0.05
SHOW_PRIVATE = False
DEVICE = best_device()
print("training on:", DEVICE)

training on: mps


## 1. Freeze the benchmark
Built once from the newest OpenPhish snapshot and today's Chrome history, then reused forever. To deliberately rebuild it later, delete `data/processed/benchmark_frozen.csv.gz`, and record the change in the journal, because old and new scores stop being comparable.

In [2]:
if FROZEN_PATH.exists():
    print("using the existing frozen benchmark")
else:
    freeze_benchmark()
    print("froze a new benchmark")
bench = load_benchmark()
print(json.loads(FROZEN_PATH.with_suffix("").with_suffix(".json").read_text())["frozen_on"])
bench.groupby(["source", "split"]).size().unstack()

froze a new benchmark
2026-10-26


split,calib,eval
source,,
history,226,172
openphish,135,165
tranco,2520,2480


## 2. Data, with and without query strings

In [3]:
data = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "fullurl_dataset.csv.gz", keep_default_na=False)
data = data[data["text"].str.len() > 0].reset_index(drop=True)
data["text_noq"] = data["text"].map(lambda t: normalize_full_url(t, drop_query=True))
splits = {s: data[data["split"] == s] for s in ["train", "val", "test"]}

has_q = data["text"].str.contains("?", regex=False)
print("share with a query string:")
print(has_q.groupby(data["source"]).mean().round(3).to_string())

# Benchmark domains that ended up in the stage 2 data can't be used fairly.
overlap = bench["registered"].isin(set(data["registered"]))
print(f"\nbenchmark rows dropped for overlapping the training data: {overlap.sum()}")
bench = bench[~overlap].reset_index(drop=True)

share with a query string:
source
commoncrawl         0.215
commoncrawl_home    0.000
phishing_db         0.160
sitemap             0.021

benchmark rows dropped for overlapping the training data: 0


### Does stripping queries lower the leak detector?
Same shape-only leak detector as notebook 06, run on the query-stripped text. If the `?` was a big part of the shortcut, this should drop from 0.891.

In [4]:
%%time
stats_noq = shallow_stats(data["text_noq"]).drop(columns=["has_query"])
leak_p = cross_val_predict(HistGradientBoostingClassifier(max_iter=200, random_state=RANDOM_STATE),
                           stats_noq, data["is_phishing"], groups=data["registered"],
                           cv=GroupKFold(n_splits=5), method="predict_proba")[:, 1]
LEAK_NOQ = average_precision_score(data["is_phishing"], leak_p)
LEAK_Q = json.loads((PROJECT_ROOT / "data" / "processed" / "fullurl_dataset.json").read_text())["leak_detector_pr_auc"]
print(f"leak detector PR-AUC: with queries {LEAK_Q:.3f}  ->  without {LEAK_NOQ:.3f}")

leak detector PR-AUC: with queries 0.891  ->  without 0.886
CPU times: user 24.9 s, sys: 6.6 s, total: 31.5 s
Wall time: 4.31 s


## 3. TF-IDF, with and without queries
Fast (seconds), and it beat the CNN in notebook 07, so it's a serious candidate.

In [5]:
%%time
tr, te = splits["train"], splits["test"]
models = {}
for name, dq in [("TF-IDF", False), ("TF-IDF no-q", True)]:
    models[name] = FullUrlTfidf(drop_query=dq).fit(tr["text"].tolist(), tr["is_phishing"].values)
    p = models[name].phish_proba(te["text"].tolist())
    print(f"{name:12s} test PR-AUC {average_precision_score(te['is_phishing'], p):.4f}")

TF-IDF       test PR-AUC 0.9805
TF-IDF no-q  test PR-AUC 0.9797
CPU times: user 15.5 s, sys: 847 ms, total: 16.4 s
Wall time: 15.4 s


## 4. Char-CNN, with and without queries
The with-query CNN is notebook 07's saved model (no retraining). The no-query CNN is trained here, same settings.

In [6]:
%%time
cnn_path = PROJECT_ROOT / "models" / "charcnn.joblib"
models["CNN"] = joblib.load(cnn_path)
models["CNN no-q"] = CharCNNClassifier(seed=RANDOM_STATE, drop_query=True).fit(
    tr["text"].tolist(), tr["is_phishing"].values,
    val_urls=splits["val"]["text"].tolist(), val_y=splits["val"]["is_phishing"].values,
    epochs=12, batch_size=256, lr=2e-3, patience=2, device=DEVICE)
for name in ["CNN", "CNN no-q"]:
    p = models[name].phish_proba(te["text"].tolist())
    print(f"{name:9s} test PR-AUC {average_precision_score(te['is_phishing'], p):.4f}")

epoch  1  loss 0.3536  val PR-AUC 0.9712  (15s)
epoch  2  loss 0.2145  val PR-AUC 0.9784  (15s)
epoch  3  loss 0.1689  val PR-AUC 0.9818  (15s)
epoch  4  loss 0.1358  val PR-AUC 0.9816  (15s)
epoch  5  loss 0.1154  val PR-AUC 0.9837  (15s)
epoch  6  loss 0.1045  val PR-AUC 0.9825  (15s)
epoch  7  loss 0.0914  val PR-AUC 0.9823  (15s)
no improvement for 2 epochs, stopping
CNN       test PR-AUC 0.9746
CNN no-q  test PR-AUC 0.9763
CPU times: user 12.3 s, sys: 2.63 s, total: 15 s
Wall time: 1min 51s


## 5. Probes
The no-query models should give the two `index.html` probes the same score, since they can't see the part after `?`.

In [7]:
probes = ["example.com", "example.com/about/team", "example.com/index.html", "example.com/index.html?id=8f3a9c2e7b1d",
          "example.com/search?q=shoes&page=2", "example.com/login.php", "example.com/wp-includes/js/auth.php",
          "example.com/blog/2024/10/hello-world"]
pd.DataFrame({"probe": probes, **{n: m.phish_proba(probes).round(3) for n, m in models.items()}})

,probe,TF-IDF,TF-IDF no-q,CNN,CNN no-q
0,example.com,0.268,0.203,0.385,0.194
1,example.com/about/team,0.035,0.027,0.004,0.001
2,example.com/index.html,0.897,0.914,0.932,0.209
3,example.com/index.html?id=8f3a9c2e7b1d,0.873,0.914,0.892,0.209
4,example.com/search?q=shoes&page=2,0.037,0.025,0.086,0.002
5,example.com/login.php,0.999,0.998,0.992,0.994
6,example.com/wp-includes/js/auth.php,0.999,0.999,1.000,1.000
7,example.com/blog/2024/10/hello-world,0.005,0.004,0.000,0.000


## 6. The frozen benchmark
Same rules as notebooks 05 and 07: each system's threshold is the lowest one with <= 5% false alarms on my history in the **calibration** half, and everything is reported on the **evaluation** half.

New column, **history FA (with ?)**: false alarms on history URLs that contain a query string. That's the direct test of the theory.

Combinations average v1's two host models with one full-URL model, then apply v1's allowlist.

In [8]:
%%time
v1 = PhishDetector.load(PROJECT_ROOT / "models" / "detector_v1.joblib")
urls = bench["url"].tolist()
allow = np.array([v1.is_allowlisted(u) for u in urls])
host = v1.model_scores(urls).sum(axis=1).values

scores = {"v1": v1.phish_proba(urls)}
for name, m in models.items():
    p = m.phish_proba(urls)
    scores[name] = p
    scores[f"v1 + {name}"] = np.where(allow, 0.0, (host + p) / 3)
for k, v in scores.items():
    bench[f"p::{k}"] = v

CPU times: user 2.9 s, sys: 749 ms, total: 3.65 s
Wall time: 2.27 s


In [9]:
def pick_threshold(legit_scores, target_fa):
    for t in np.sort(np.unique(np.r_[legit_scores, 1.0])):
        if (legit_scores >= t).mean() <= target_fa:
            return float(t)
    return 1.0

def rate(mask, flagged):
    return flagged[mask].mean() if mask.any() else np.nan

src = bench["source"]
cal, ev = bench["split"] == "calib", bench["split"] == "eval"
tail = bench["rank"] > 10_000
hist_q = (src == "history") & bench["url"].str.contains("?", regex=False)
LEGIT = "history" if (src == "history").any() else "tranco"

rows, thresholds = [], {}
for s in scores:
    p = bench[f"p::{s}"]
    t = pick_threshold(p[cal & (src == LEGIT)].values, TARGET_FA)
    thresholds[s] = t
    f = p >= t
    rows.append({
        "system": s, "threshold": t,
        "history FA": rate(ev & (src == "history"), f),
        "history FA (with ?)": rate(ev & hist_q, f),
        "Tranco tail FA": rate(ev & (src == "tranco") & tail, f),
        "recall": rate(ev & (src == "openphish"), f),
        "recall (platform)": rate(ev & (src == "openphish") & bench["platform"], f),
        "recall (other)": rate(ev & (src == "openphish") & ~bench["platform"], f),
    })
scoreboard = pd.DataFrame(rows).set_index("system").round(3)
print(f"eval half: {(ev & (src == 'history')).sum()} history URLs ({(ev & hist_q).sum()} with '?'), "
      f"{(ev & (src == 'openphish')).sum()} OpenPhish URLs")
scoreboard

eval half: 172 history URLs (72 with '?'), 165 OpenPhish URLs


,threshold,history FA,history FA (with ?),Tranco tail FA,recall,recall (platform),recall (other)
system,,,,,,,
v1,0.310,0.058,0.056,0.301,0.915,1.000,0.803
TF-IDF,0.993,0.029,0.069,0.002,0.188,0.298,0.042
v1 + TF-IDF,0.507,0.035,0.042,0.174,0.861,1.000,0.676
TF-IDF no-q,0.968,0.041,0.083,0.012,0.539,0.713,0.310
v1 + TF-IDF no-q,0.473,0.041,0.042,0.214,0.879,1.000,0.718
CNN,1.000,0.029,0.069,0.000,0.176,0.266,0.056
v1 + CNN,0.526,0.023,0.014,0.140,0.830,0.989,0.620
CNN no-q,0.998,0.017,0.028,0.007,0.436,0.553,0.282
v1 + CNN no-q,0.375,0.047,0.042,0.237,0.891,1.000,0.746


How to read it:

- **Threshold near 1.0** means the model had to be almost switched off to stay under 5% false alarms. That's what happened to notebook 07's CNN.
- **Compare each model with its no-q version.** If stripping queries fixed the shortcut, the no-q threshold drops, history FA (with ?) drops, and recall rises.
- **Compare v1 + X with v1.** A combination is only worth shipping if it beats v1's recall, especially recall (other), at similar false alarms.
- With about 150 to 170 URLs per group in the evaluation half, differences of a few points are noise.

## 7. Against v1: what the best combination gains and loses

In [10]:
combos = [s for s in scores if s.startswith("v1 + ")]
best = scoreboard.loc[combos, "recall"].idxmax()
print("best combination by recall:", best)
v1_flag = bench["p::v1"] >= thresholds["v1"]
b_flag = bench[f"p::{best}"] >= thresholds[best]
op = ev & (src == "openphish")
gained = bench[op & ~v1_flag & b_flag][["url", "platform", "p::v1", f"p::{best}"]].copy()
lost = bench[op & v1_flag & ~b_flag][["url", "platform", "p::v1", f"p::{best}"]].copy()
for d in (gained, lost):
    d["url"] = d["url"].map(defang)
print(f"newly caught: {len(gained)}   newly missed: {len(lost)}")
display(gained.head(15)); display(lost.head(15))

best combination by recall: v1 + CNN no-q
newly caught: 4   newly missed: 8


,url,platform,p::v1,p::v1 + CNN no-q
5454,hxxps://sultoneff[.]id/wp-admin/owa_logon/,False,0.276149,0.517392
5496,hxxp://busardoh[.]com/wp-content/lblogin/login,False,0.213392,0.475358
5601,hxxps://wh-appies[.]com/index[.]html,False,0.170483,0.426978
5605,hxxps://wh-appevs[.]com/index[.]html,False,0.266888,0.492229


,url,platform,p::v1,p::v1 + CNN no-q
5435,hxxp://roblox[.]com[.]gr/,False,0.342454,0.283961
5464,hxxps://www[.]roblox[.]ss/communities/2886525628/Robux-Studios,False,0.448049,0.298940
5510,hxxps://www[.]roblox[.]rw/games/14930532786/98xx-1M-VISITS?privateServerLinkCode=1104029018576654500574631...,False,0.397567,0.265396
5511,hxxps://www[.]roblox[.]rw/games/140242266549025/The-Whistle-Occurrence?privateServerLinkCode=1104029018576...,False,0.397567,0.266328
5512,hxxps://www[.]roblox[.]rw/games/81099268327915/mihecraft?privateServerLinkCode=110402901857665450057463129...,False,0.397567,0.284212
5594,hxxp://subscribe-spotify[.]com/,False,0.328826,0.356263
5613,hxxps://www[.]roblox[.]com[.]bn/games/2753915549/Blox-Fruits?privateServerLinkCode=47802401840295769168104...,False,0.441488,0.298257
5677,hxxp://rbxshader[.]com/,False,0.351504,0.344425


In [11]:
h = bench[src == "history"]
if SHOW_PRIVATE:
    display(h.nlargest(15, f"p::{best}")[["url", "p::v1", f"p::{best}"]])
else:
    print(f"{len(h)} history URLs. Flagged: v1 {(h['p::v1'] >= thresholds['v1']).sum()}, "
          f"{best} {(h[f'p::{best}'] >= thresholds[best]).sum()}")

398 history URLs. Flagged: v1 21, v1 + CNN no-q 18


## 8. Save
The no-query CNN, the scoreboard, and the best combination as a **candidate** detector. The API keeps serving v1 unless the scoreboard says the candidate is clearly better.

In [12]:
from datetime import date
joblib.dump(models["CNN no-q"], PROJECT_ROOT / "models" / "charcnn_noq.joblib", compress=3)
(PROJECT_ROOT / "reports").mkdir(exist_ok=True)
scoreboard.assign(notebook="08", date=date.today().isoformat(), target_fa=TARGET_FA).to_csv(
    PROJECT_ROOT / "reports" / "scoreboard_08.csv")

name = best.removeprefix("v1 + ")
cand = PhishDetector(models={**v1.models, name: models[name]}, allowlist=v1.allowlist,
                     threshold=thresholds[best], version="v2-candidate")
r = scoreboard.loc[best]
cand.metadata = {"combination": best, "benchmark": "frozen", "threshold_policy":
                 f"lowest threshold with <= {TARGET_FA:.0%} false alarms on {LEGIT} (calibration half)",
                 **{f"eval_{k.replace(' ', '_')}": r[k] for k in ["history FA", "Tranco tail FA", "recall", "recall (other)"]}}
cand.save(PROJECT_ROOT / "models" / "detector_v2.joblib")
print("saved candidate:", best)
print(scoreboard.loc[["v1", best]].T)

saved candidate: v1 + CNN no-q
system                  v1  v1 + CNN no-q
threshold            0.310          0.375
history FA           0.058          0.047
history FA (with ?)  0.056          0.042
Tranco tail FA       0.301          0.237
recall               0.915          0.891
recall (platform)    1.000          1.000
recall (other)       0.803          0.746


## Results log

| | leak detector |
|---|---|
| with queries | |
| without queries | |

| system | threshold | history FA | history FA (with ?) | recall | recall (other) |
|---|---|---|---|---|---|
| v1 | | | | | |
| TF-IDF / no-q | | | | | |
| CNN / no-q | | | | | |
| best v1 + X | | | | | |

Questions for the journal:
1. Did stripping queries remove the shortcut? (Probes, history FA with ?, thresholds.)
2. Does any combination beat v1 at the same false-alarm budget?
3. Is the CNN worth it over TF-IDF?